In [2]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"

In [3]:
# Helper functions
from anthropic.types import Message

# Magic string to trigger redacted thinking
thinking_test_str = "ANTHROPIC_MAGIC_STRING_TRIGGER_REDACTED_THINKING_46C9A13E193C177646C7398A98432ECCCE4C1253D5E2D82641AC0E52CC2876CB"


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    thinking = False,
    thinking_budget=1024,   # Min budget is 1024 (tokens)
):
    params = {
        "model": model,
        "max_tokens": 4000,     # Should be significantly higher than thinking_budget for text response
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if thinking:
        params["thinking"] = {
            "type": "enabled",
            "budget_tokens": thinking_budget
        }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

Note the Signature in the ThinkingBlock.  That's to prevent programmers from modifying the assistant
response text to produce dangerous behavior.
Sometimes you may get a redacted_thinking block.  You can force this with the magic thinking_test_str variable string.  Useful for testing.

In [4]:
messages = []

add_user_message(
    messages, 
    "Write a one page paragraph guide to recursion"
    )

chat(messages, thinking=True)

Message(id='msg_011Ce948T77L2MuCerkca2Rz', container=None, content=[ThinkingBlock(signature='EswFCpQBCBAYAipA3La4xHZWJf5uTCK/M1E7yZVDovMTmvE5Y1zfbn6wxqDsaUqjfKULV8zRCfRSZ6pO+/NThXVvaYILTfnMF5fREDIaY2xhdWRlLXNvbm5ldC00LTUtMjAyNTA5Mjk4AEIIdGhpbmtpbmdaJDYyNjJiOTMzLTg2YjgtNDU1Ny1hZDQ1LTZkODIzOTMyNjgzORIM0Jwj8GQ6mhxYaiMMGgwK9+3Zx5OKIHi7JKAiMEQaVpPo2+g6kSbBtmT/9aT3npjl5+QMyuOS1tcOHzuT3XNXQiMTYejYeFsr0Sbt8irkA3SSrgAmYO1OYJLFZmxTbP+wGa/SoiItRxiWs3Ikr4/7mBHQ980LthISVOmgnBYuQkWIPjkbWUXcuB6s7CwsBMkkhYFtN35Mpw31/uBDZMaMZW3P7Ii6J5Lx6LVzArlo60o6onWJO5sFn3WQ5FoCCl103/ulj1xkpUaZZww5LnpXLjl9P3zXaqcAUFw/be8QBg7FjbeibAMbfUsUA3gyAGpgllYbsneHTvHA/iiALyLGG8t/kCo+S/6NzBoShIKKGbE+s+hgSSeZBbMiyKUGReO+JuBi3eAT4FXVxx1n1NRKBG9l47sv6lrxna0FYk9MNn+EICYWB/pDoiZ6qV7x0o077rWy0/nZd/5G+kiJxu4Kjp8d1K2YHa7EvxNsyXq4hN+ee2V3YwNY8gUHa2qOI44Upgmme3qVGWog/SEYC+fPQ7qw5oGT3H9mea5qesRwDrY9h9/GbdR22h8Ac4ztkp5EFSS65DsDue2YFEGOsgfUEJmPnYTpCHZV9ZeQSB5QFkMuNWpG2vvfLGLtXzgmAIiit4iO2IrlYWPK37PNs8yiI1VY9qS3rjFB5NlqwarUOGmzHsJIj9jdA/1jufO

In [5]:
messages = []

add_user_message(
    messages, 
    thinking_test_str
    )

chat(messages, thinking=True)

Message(id='msg_011Ce94UG5tNMFG23a6ASdN5', container=None, content=[RedactedThinkingBlock(data='EpAFCpQBCBAYAipA8DxcHvzGzNWIjRSb0sVIzGTnWFYNjYRAM2jrSXejTFuElCCuot8VoSZMGF7jig5TTszs43rbC+RrW9KXdK6ZwDIaY2xhdWRlLXNvbm5ldC00LTUtMjAyNTA5Mjk4AEIIdGhpbmtpbmdaJDYyNjJiOTMzLTg2YjgtNDU1Ny1hZDQ1LTZkODIzOTMyNjgzORIMr2XWuc7jn5+vKcwlGgwLOb3JWZwZLGqaCz8iMCIpHYXR0NC7FZ9MnOGY8wwtjuzndu+rUwsQwuNQQaWf8Y/pyggYTp6DUG2ues99qSqoAw3EEi2Tyra8CMn64esPI+u8upn9Ur6ufCxwHkuWUoLkWV2gFDs5F54RyxZ+DG972zTHdZEuijRsZIUbLFd+tLU4EBV8TTGvDNRG/7TVKhMciH0W41qgQA8VpzZp3d1Y0cDnOLLgxWfbGKYbuZLeFJwb3kIdw7hmUWWAS0y3Ap63wKMRbXQrTW78i9wgnhbXJxRzyFJd0JAs4dlEoO/foWBpcqvOA9TYe5BzWLvE2JQrJ3mA4HVbvd+3qjmj3S2fyZlzixHqjNbSW+0D0dn8smKdrgLdvEIgjkDsKM+QvOgqhyqcEcvprnRnprJUiuj7F44QEYLiT6tOipCVjOCp30A2ZFQydraYaI8i6sZC9bPZanvVpV6QcSmFKu4yqnVLImwxtsmMY9u05T9/v5yPeTSqQonBhp8a5ubP+rlLEBJEezNo4NGqu9vbK30RAwjRRYqXMP/7XNp+a6llSxOhTHib5oOHot7lPX1HWQ8S6CYqsEYG8+MG7JxM7y2DWMh3++RV4/wUx5N6H9g3YuQYtQQwJStTUSsXYonp+QvROoqfQsK/yULJe7IYAQ==', type='redacted_th